# **Assignment 4 - UCS420 Cognitive Computing**
**Name:** Harnoor Kaur<br/>
**Roll No:** 1024170436 

## Q1: Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your own roll number digits

In [1]:
import pandas as pd
roll_number = input("Enter your college roll number: ")

last_two_digits = roll_number[-2:]

print("Last two digits:", last_two_digits)

Last two digits: 


In [2]:
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": ["fee cost price charge"],
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Security > Reset Password.",
        "keywords": ["password reset login"],
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": ["hours timing open time"],
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": ["pay payment upi fee"],
        "category": "billing"
    }
]

In [3]:
category_list = ["billing", "account", "general"]

personalized_entries = []

question_options = {
    "billing": [
        "how can i check my payment details",
        "how do i get a receipt for my payment",
        "how can i update my billing information"
    ],
    
    "account": [
        "how do i update my registered mobile number",
        "how can i change my account details",
        "how do i update my email address"
    ],
    
    "general": [
        "where can i find general information",
        "how can i contact customer support",
        "where can i check the available services"
    ]
}

keyword_options = {
    "billing": [
        "payment bill transaction",
        "receipt payment invoice",
        "billing payment account"
    ],
    
    "account": [
        "mobile number update",
        "account details change",
        "email address update"
    ],
    
    "general": [
        "information details help",
        "support contact assistance",
        "services information available"
    ]
}

for i, digit in enumerate(last_two_digits):
    digit = int(digit)
    
    category = category_list[digit % 3]
    
    question = question_options[category][digit % len(question_options[category])]
    keywords = keyword_options[category][digit % len(keyword_options[category])]
    
    personalized_entries.append({
        "question": question,
        "answer": "Please refer to the relevant section for more information.",
        "keywords": [keywords],
        "category": category
    })

personalized_entries

[]

In [4]:
all_entries = fixed_entries + personalized_entries

df = pd.DataFrame(all_entries)

print(df)

                      question                                       answer  \
0       what is the annual fee                    The annual fee is Rs 500.   
1        how to reset password  Go to Settings > Security > Reset Password.   
2  what are your working hours                    We are open 9 AM to 5 PM.   
3        how can i pay the fee   You can pay via UPI, card, or net banking.   

                   keywords category  
0   [fee cost price charge]  billing  
1    [password reset login]  account  
2  [hours timing open time]  general  
3     [pay payment upi fee]  billing  


## Q2: Generate and Score a Hypothesis

In [5]:
def score_query(query, dataframe):
    query_words = set(query.lower().split())
    
    results = []
    
    for index, row in dataframe.iterrows():
        # Convert keyword string into individual words
        keyword_words = set(row["keywords"][0].lower().split())
        
        matched_words = query_words.intersection(keyword_words)
        
        score = len(matched_words)
        
        if len(keyword_words) > 0:
            confidence = score / len(keyword_words)
        else:
            confidence = 0
        
        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": confidence
            })
    
    # Sort by confidence in descending order
    results.sort(key=lambda x: x["confidence"], reverse=True)
    
    return results

In [6]:
query = input("Enter your query: ")

results = score_query(query, df)

for result in results:
    print("Question:", result["question"])
    print("Category:", result["category"])
    print("Confidence:", result["confidence"])
    print()

## Q3: Find FAQs Belonging to a Category

In [7]:
def same_category(category_name, df):
    category_name = category_name.lower()
    
    result = df[df["category"].str.lower() == category_name]
    
    return result["question"].tolist()

In [8]:
category = input("Enter a category (billing/account/general): ")

questions = same_category(category, df)

print("Questions in", category, "category:")

for question in questions:
    print("-", question)

Questions in  category:


## Q4: Update an FAQ Entry

In [9]:
print("Current FAQ entries:")

for i, question in enumerate(df["question"]):
    print(i, "-", question)

entry_number = int(input("Enter the entry number to update: "))
new_keyword = input("Enter a new keyword: ")

# Get the current keywords
current_keywords = df.at[entry_number, "keywords"][0]

# Add the new keyword
df.at[entry_number, "keywords"] = [
    current_keywords + " " + new_keyword
]

print("\nUpdated DataFrame:")
print(df)

Current FAQ entries:
0 - what is the annual fee
1 - how to reset password
2 - what are your working hours
3 - how can i pay the fee


ValueError: invalid literal for int() with base 10: ''

## Q5: Count FAQs in Each Category

In [ ]:
category_counts = df.groupby("category").size()
print("Number of FAQs in each category:")
print(category_counts)

Number of FAQs in each category:
category
account    1
billing    4
general    1
dtype: int64


## Q6: Handle Ties in Confidence Scores

In [ ]:
def score_query_with_ties(query, dataframe):
    query_words = set(query.lower().split())
    
    results = []
    
    for index, row in dataframe.iterrows():
        keyword_words = set(row["keywords"][0].lower().split())
        
        matched_words = query_words.intersection(keyword_words)
        
        score = len(matched_words)
        
        if len(keyword_words) > 0:
            confidence = score / len(keyword_words)
        else:
            confidence = 0
        
        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": confidence
            })
    
    if not results:
        return []
    

    highest_confidence = max(
        result["confidence"] for result in results
    )
    
    tied_results = [
        result for result in results
        if result["confidence"] == highest_confidence
    ]
    
    return tied_results

### Testing Q6

Two test cases are used:

1. A query containing `"fee"` should match both billing-related FAQs.
2. A query that does not match any FAQ should produce no result.

In [ ]:
print("Test 1: Query = 'fee'")

results = score_query_with_ties("fee", df)

for result in results:
    print("Question:", result["question"])
    print("Category:", result["category"])
    print("Confidence:", result["confidence"])
    print()

Test 1: Query = 'fee'
Question: how can i pay the fee
Category: billing
Confidence: 0.25



In [ ]:
print("Test 2: Query = 'xyzabc'")

results = score_query_with_ties("xyzabc", df)

if not results:
    print("No matching FAQ found.")
else:
    for result in results:
        print(result)

Test 2: Query = 'xyzabc'
No matching FAQ found.
